# Example 2: Simple CNN with MNIST

In [ ]:
!pip install torch torchvision matplotlib numpy scikit-learn tqdm

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader
import torchvision
from torchvision.datasets import MNIST
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import tqdm

# 0. Computation of MNIST mean and standard deviation

In [2]:
dataset = MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())

In [ ]:
data_loader = DataLoader(dataset, batch_size=128, shuffle=False)

# Initialize sums
n_pixels = 0
sum_ = 0.0
sum_squared = 0.0

for images, _ in tqdm.tqdm(data_loader):
    # images shape: [B, C, H, W]
    batch_samples = images.size(0)  # batch size (B)
    n_pixels += batch_samples * 28 * 28  # since C=1
    sum_ += images.sum().item()
    sum_squared += (images ** 2).sum().item()

# Compute mean and std
mean = sum_ / n_pixels
std = (sum_squared / n_pixels - mean ** 2) ** 0.5

In [ ]:
print(f'{mean=:.4f}, {std=:.4f}')

In [5]:
del dataset, data_loader, images

# 1. Data Preparation

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean, ), (std, ))
])

In [7]:
train_set = MNIST(root='./data', train=True, download=True, transform=transform)
train_set, val_set = random_split(train_set, lengths=[0.8, 0.2])

In [8]:
test_set = MNIST(root='./data', train=False, download=True, transform=transform)

In [9]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False, drop_last=False)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, drop_last=False)

In [ ]:
# images and labels
input, target = next(iter(train_loader))
print('target:', ' '.join(str(each.item()) for each in target[:8]))

In [ ]:
img = torchvision.utils.make_grid(input[:8], nrow=8)
img = img * 0.3081 + 0.1307  # unnormalize
np_img = img.numpy()
plt.imshow(np.transpose(np_img, (1, 2, 0)), cmap='gray')
plt.axis('off')
plt.show()

# 2. Define CNN Model

In [16]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)
        self.pool  = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5)
        self.fc1   = nn.Linear(32 * 4 * 4, 64)
        self.fc2   = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32 * 4 * 4)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 3. Training

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

In [ ]:
best_val_loss = float('inf')
patience = 3
counter = 0

for epoch in tqdm.trange(5):
    running_loss = 0.0
    model.train()
    for input, target in train_loader:
        input, target = input.to(device), target.to(device)
        optimizer.zero_grad()
        outputs = model(input)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Training Loss: {running_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for input, target in val_loader:
            input, target = input.to(device), target.to(device)
            outputs = model(input)
            loss = criterion(outputs, target)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    avg_val_loss = val_loss / len(val_loader)
    print(f"Validation Loss: {avg_val_loss:.4f}, Accuracy: {100 * correct / total:.2f}%")

    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

## 4. Evaluation

In [27]:
## Load best model
model.load_state_dict(torch.load('best_model.pth'))

<All keys matched successfully>

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for input, target in test_loader:
        input, target = input.to(device), target.to(device)
        outputs = model(input)
        _, predicted = torch.max(outputs, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f'Accuracy on test set: {100 * correct / total:.2f}%')

Accuracy on test set: 99.03%


## 5. Visualizing hidden feature maps

In [25]:
image = input[0].unsqueeze(0).to(device)
with torch.inference_mode():
    conv1_out = model.pool(torch.relu(model.conv1(image)))
    conv2_out = model.pool(torch.relu(model.conv2(conv1_out)))

In [ ]:
fig, axarr = plt.subplots(1, 10, figsize=(15, 3))

example_idx = 0

for idx in range(10):
    axarr[idx].imshow(conv1_out[example_idx, idx].cpu(), cmap='gray')
    axarr[idx].axis('off')
fig.suptitle('Conv1 Feature Maps')
plt.show()

fig, axarr = plt.subplots(1, 10, figsize=(15, 3))
for idx in range(10):
    axarr[idx].imshow(conv2_out[example_idx, idx].cpu(), cmap='gray')
    axarr[idx].axis('off')
fig.suptitle('Conv2 Feature Maps')
plt.show()

## 6. Confusion matrix

In [22]:
y_pred = []
y_true = []
with torch.inference_mode():
    for input, target in tqdm.tqdm(test_loader):
        input = input.to(device)
        output = model(input)
        _, prediction = torch.max(output, dim=1)
        y_pred.extend(prediction.cpu().numpy())
        y_true.extend(target.cpu().numpy())

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 79/79 [00:02<00:00, 31.10it/s]


In [23]:
cm = confusion_matrix(y_true=y_true, y_pred=y_pred, normalize='true')

In [24]:
cm_plot = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=range(10)
)

In [ ]:
cm_plot.plot(values_format='.2f')